In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .config('spark.driver.memory', '8g') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 11:47:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [14]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-04 12:00:24--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2462:4c00:b:20a5:b140:21, 2600:9000:2462:1a00:b:20a5:b140:21, 2600:9000:2462:7400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2462:4c00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  51.8MB/s    in 5.7s    

2026-03-04 12:00:29 (51.6 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [15]:
df = spark.read \
    .option("header", "true") \
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [16]:
df.repartition(6)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp_ntz, on_scene_datetime: timestamp_ntz, pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: bigint, DOLocationID: bigint, trip_miles: double, trip_time: bigint, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string]

In [17]:
df.write.parquet('fhvhv/2021/01/', mode = 'overwrite')

In [27]:
df = spark.read.parquet('fhvhv/2026/01/')

In [28]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [33]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2026-01-04 07:46:13|2026-01-04 07:55:19|          61|         188|
|2026-01-04 07:12:19|2026-01-04 08:02:50|          50|         265|
|2026-01-04 07:19:23|2026-01-04 07:31:28|          89|         149|
|2026-01-04 07:34:17|2026-01-04 08:14:17|         149|         138|
|2026-01-04 07:06:56|2026-01-04 07:15:21|         226|         145|
|2026-01-04 07:26:35|2026-01-04 07:40:43|         145|         138|
|2026-01-04 07:52:09|2026-01-04 08:05:32|         260|           4|
|2026-01-04 07:35:18|2026-01-04 07:53:18|         177|         138|
|2026-01-04 07:37:28|2026-01-04 07:58:24|         112|          75|
|2026-01-04 07:45:16|2026-01-04 07:57:23|         215|         121|
|2026-01-04 07:13:34|2026-01-04 07:21:49|         225|          61|
|2026-01-04 07:23:10|2026-01-04 07:43:25|       

In [34]:
from pyspark.sql import functions as F

In [37]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'pickup_datetime', 'dropoff_date', 'dropoff_datetime') \
    .show()

+-----------+-------------------+------------+-------------------+
|pickup_date|    pickup_datetime|dropoff_date|   dropoff_datetime|
+-----------+-------------------+------------+-------------------+
| 2026-01-04|2026-01-04 07:46:13|  2026-01-04|2026-01-04 07:55:19|
| 2026-01-04|2026-01-04 07:12:19|  2026-01-04|2026-01-04 08:02:50|
| 2026-01-04|2026-01-04 07:40:49|  2026-01-04|2026-01-04 07:54:43|
| 2026-01-04|2026-01-04 07:19:23|  2026-01-04|2026-01-04 07:31:28|
| 2026-01-04|2026-01-04 07:34:17|  2026-01-04|2026-01-04 08:14:17|
| 2026-01-04|2026-01-04 07:06:56|  2026-01-04|2026-01-04 07:15:21|
| 2026-01-04|2026-01-04 07:26:35|  2026-01-04|2026-01-04 07:40:43|
| 2026-01-04|2026-01-04 07:52:09|  2026-01-04|2026-01-04 08:05:32|
| 2026-01-04|2026-01-04 07:43:30|  2026-01-04|2026-01-04 07:47:56|
| 2026-01-04|2026-01-04 07:53:44|  2026-01-04|2026-01-04 07:58:30|
| 2026-01-04|2026-01-04 07:35:18|  2026-01-04|2026-01-04 07:53:18|
| 2026-01-04|2026-01-04 07:37:28|  2026-01-04|2026-01-04 07:58

In [38]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [39]:
crazy_stuff('B02884')

's/b44'

In [41]:
from pyspark.sql import types
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [43]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'pickup_datetime', 'dropoff_date', 'dropoff_datetime') \
    .show()

[Stage 16:>                                                         (0 + 1) / 1]

+-------+-----------+-------------------+------------+-------------------+
|base_id|pickup_date|    pickup_datetime|dropoff_date|   dropoff_datetime|
+-------+-----------+-------------------+------------+-------------------+
|  e/d4c| 2026-01-04|2026-01-04 07:46:13|  2026-01-04|2026-01-04 07:55:19|
|  e/d4c| 2026-01-04|2026-01-04 07:12:19|  2026-01-04|2026-01-04 08:02:50|
|  e/d4e| 2026-01-04|2026-01-04 07:40:49|  2026-01-04|2026-01-04 07:54:43|
|  e/d4c| 2026-01-04|2026-01-04 07:19:23|  2026-01-04|2026-01-04 07:31:28|
|  e/d4c| 2026-01-04|2026-01-04 07:34:17|  2026-01-04|2026-01-04 08:14:17|
|  e/d4c| 2026-01-04|2026-01-04 07:06:56|  2026-01-04|2026-01-04 07:15:21|
|  e/d4c| 2026-01-04|2026-01-04 07:26:35|  2026-01-04|2026-01-04 07:40:43|
|  e/d4c| 2026-01-04|2026-01-04 07:52:09|  2026-01-04|2026-01-04 08:05:32|
|  e/d4e| 2026-01-04|2026-01-04 07:43:30|  2026-01-04|2026-01-04 07:47:56|
|  e/d4e| 2026-01-04|2026-01-04 07:53:44|  2026-01-04|2026-01-04 07:58:30|
|  e/d4c| 2026-01-04|2026